In [ ]:
# Cell 1 — locate the saved simulation inputs and define the conditional RealNVP.
from pathlib import Path
import json, math, time, random, gc
import numpy as np
import pandas as pd
import torch
from torch import nn

INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/day16-flow-calibration')
OUT.mkdir(parents=True, exist_ok=True)

found = {'binary': [], 'zeeman': []}
for meta_path in INPUT.rglob('simulation_metadata.json'):
    try:
        meta = json.loads(meta_path.read_text(encoding='utf-8'))
        family = meta.get('family')
        root = meta_path.parent
        if family in found and (root/'parameters.csv').is_file() and (root/'observations.npy').is_file():
            found[family].append((int(meta.get('parameter_draws_unique', 0)), root, meta))
    except Exception:
        pass

SIMS = {}
for family in ('binary', 'zeeman'):
    if not found[family]:
        raise FileNotFoundError(
            f'Missing saved Day 7 {family} simulations. Attach the successful Cosmic Swan Bayesian '
            f'output containing simulations/{family}/observations.npy, parameters.csv, and simulation_metadata.json.'
        )
    found[family].sort(key=lambda row: (row[0], str(row[1])))
    draws, root, meta = found[family][-1]
    if draws < 100_000:
        print(f'[WARNING] {family} has only {draws:,} unique draws in the selected input.')
    SIMS[family] = (root, meta, draws)
    print(f'[PASS] {family}: {root} | unique prior draws={draws:,}')

PARAMETER_SPECS = {
    'binary': [
        ('period_days', 0.1, 13.0, 'log'), ('phase', 0.0, 1.0, 'linear'),
        ('depth_primary', 0.002, 0.45, 'linear'), ('depth_ratio', 0.0, 1.0, 'linear'),
        ('duration_fraction', 0.005, 0.12, 'linear'), ('ellipsoidal_amplitude', 0.0, 0.02, 'linear'),
        ('reflection_amplitude', -0.01, 0.01, 'linear'), ('noise_sigma', 1e-4, 0.03, 'log'),
    ],
    'zeeman': [
        ('field_gauss', 0.0, 20_000.0, 'linear'), ('radial_velocity_km_s', -250.0, 250.0, 'linear'),
        ('line_depth', 0.1, 0.85, 'linear'), ('line_sigma_angstrom', 0.25, 1.5, 'linear'),
        ('noise_sigma', 5e-4, 0.04, 'log'),
    ],
}

def to_unconstrained(theta, specs):
    cols = []
    for j, (_, low, high, scale) in enumerate(specs):
        if scale == 'log':
            low, high = np.log(low), np.log(high)
            val = np.log(np.maximum(theta[:, j], np.exp(low)))
        else:
            val = theta[:, j]
        unit = np.clip((val-low)/(high-low), 1e-5, 1-1e-5)
        cols.append(np.log(unit)-np.log1p(-unit))
    return np.stack(cols, axis=1).astype(np.float32)

def from_unconstrained(values, specs):
    unit = 1.0/(1.0+np.exp(-np.clip(values, -40, 40)))
    cols=[]
    for j, (_, low, high, scale) in enumerate(specs):
        if scale == 'log': low, high = np.log(low), np.log(high)
        val = low + unit[:, :, j]*(high-low)
        cols.append(np.exp(val) if scale == 'log' else val)
    return np.stack(cols, axis=-1)

def make_flow_class():
    class Coupling(nn.Module):
        def __init__(self, dimension, context_dimension, mask):
            super().__init__()
            self.register_buffer('mask', torch.as_tensor(mask, dtype=torch.float32))
            self.network = nn.Sequential(
                nn.Linear(dimension+context_dimension, 256), nn.GELU(),
                nn.Linear(256, 256), nn.GELU(), nn.Linear(256, 2*dimension),
            )
        def params_for(self, values, context):
            mask = self.mask.expand_as(values)
            shift, raw_scale = self.network(torch.cat((values*mask, context), dim=-1)).chunk(2, dim=-1)
            active = 1-mask
            return shift*active, (1.5*torch.tanh(raw_scale/1.5))*active
        def inverse(self, values, context):
            shift, log_scale = self.params_for(values, context)
            active = 1-self.mask
            out = values*self.mask + active*((values-shift)*torch.exp(-log_scale))
            return out, -log_scale.sum(dim=-1)
        def forward(self, values, context):
            shift, log_scale = self.params_for(values, context)
            active = 1-self.mask
            return values*self.mask + active*(values*torch.exp(log_scale)+shift)
    class ConditionalRealNVP(nn.Module):
        def __init__(self, dimension, context_dimension, layers=8):
            super().__init__()
            masks=[np.asarray([(j+layer)%2 for j in range(dimension)], dtype=np.float32) for layer in range(layers)]
            self.layers=nn.ModuleList([Coupling(dimension, context_dimension, mask) for mask in masks])
        def log_prob(self, values, context):
            logdet=torch.zeros(values.shape[0], device=values.device)
            current=values
            for layer in reversed(self.layers):
                current, det=layer.inverse(current, context)
                logdet=logdet+det
            logbase=-0.5*(current.square()+math.log(2*math.pi)).sum(dim=-1)
            return logbase+logdet
        def sample(self, base, context):
            current=base
            for layer in self.layers:
                current=layer(current, context)
            return current
    return ConditionalRealNVP

ConditionalRealNVP = make_flow_class()
print('[PASS] Conditional RealNVP defined; model families:', ', '.join(PARAMETER_SPECS))


In [ ]:
# Cell 2 — train, conformal-calibrate, and test each saved family in turn.
import matplotlib.pyplot as plt

SEED = 1601
EPOCHS = 60
BATCH_SIZE = 512
PATIENCE = 8
POSTERIOR_DRAWS = 256
CASE_BATCH = 16
ALPHA = 0.10
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('[INFO] PyTorch device:', DEVICE, '| CUDA devices:', torch.cuda.device_count())

def posterior_quantiles(model, context, indices, specs, x_scaled, y_mean, y_std, draws, seed):
    model.eval()
    d=len(specs)
    q05=np.empty((len(indices),d), dtype=np.float32)
    q50=np.empty_like(q05); q95=np.empty_like(q05)
    generator=torch.Generator(device=DEVICE).manual_seed(seed)
    with torch.no_grad():
        for start in range(0,len(indices),CASE_BATCH):
            batch_idx=indices[start:start+CASE_BATCH]
            cond=torch.from_numpy(x_scaled[batch_idx]).to(DEVICE)
            b=len(batch_idx)
            base=torch.randn((b,draws,d), generator=generator, device=DEVICE)
            tiled=cond[:,None,:].expand(-1,draws,-1).reshape(b*draws,-1)
            raw=model.sample(base.reshape(b*draws,d), tiled).cpu().numpy().reshape(b,draws,d)
            raw=raw*y_std+y_mean
            physical=from_unconstrained(raw, specs)
            q05[start:start+b]=np.quantile(physical,0.05,axis=1)
            q50[start:start+b]=np.quantile(physical,0.50,axis=1)
            q95[start:start+b]=np.quantile(physical,0.95,axis=1)
            if (start//CASE_BATCH+1)%100==0:
                print(f'  posterior intervals: {min(start+b,len(indices)):,}/{len(indices):,}')
    return q05,q50,q95

all_rows=[]
summaries={}
for family_index, family in enumerate(('binary','zeeman')):
    family_start=time.time()
    root, meta, n_draws=SIMS[family]
    specs=PARAMETER_SPECS[family]
    frame=pd.read_csv(root/'parameters.csv')
    obs=np.load(root/'observations.npy', mmap_mode='r', allow_pickle=False)
    names=[s[0] for s in specs]
    missing=sorted((set(names)|{'parameter_id'}).difference(frame.columns))
    if missing: raise ValueError(f'{family} parameter table missing {missing}')
    if obs.ndim!=2 or len(obs)!=len(frame) or not len(frame):
        raise ValueError(f'{family} observations/parameters are not aligned: {obs.shape}, {len(frame)}')
    theta=frame[names].to_numpy(dtype=np.float64)
    context=np.asarray(obs, dtype=np.float32)
    if not np.isfinite(theta).all() or not np.isfinite(context).all():
        raise ValueError(f'{family} contains non-finite simulation values')
    if any((theta[:,j] <= 0).any() for j,s in enumerate(specs) if s[3]=='log'):
        raise ValueError(f'{family} log-scaled parameters must be positive')
    group_ids=frame['parameter_id'].to_numpy()
    unique_groups=np.unique(group_ids)
    rng=np.random.default_rng(SEED+family_index)
    perm=rng.permutation(unique_groups)
    ntr=int(0.70*len(perm)); nv=int(0.10*len(perm)); nc=int(0.10*len(perm))
    g_train=perm[:ntr]; g_valid=perm[ntr:ntr+nv]; g_cal=perm[ntr+nv:ntr+nv+nc]; g_test=perm[ntr+nv+nc:]
    train_idx=np.flatnonzero(np.isin(group_ids,g_train))
    valid_idx=np.flatnonzero(np.isin(group_ids,g_valid))
    cal_idx=np.flatnonzero(np.isin(group_ids,g_cal))
    test_idx=np.flatnonzero(np.isin(group_ids,g_test))
    print()
    print(f'=== {family.upper()} ===')
    print(f'[PASS] {n_draws:,} saved parameter draws; group split train/valid/calibration/test='
          f'{len(g_train):,}/{len(g_valid):,}/{len(g_cal):,}/{len(g_test):,}')

    target=to_unconstrained(theta,specs)
    y_mean=target[train_idx].mean(axis=0,keepdims=True)
    y_std=target[train_idx].std(axis=0,keepdims=True)
    y_std=np.where(y_std<1e-6,1.0,y_std).astype(np.float32)
    target=((target-y_mean)/y_std).astype(np.float32)
    x_mean=context[train_idx].mean(axis=0,keepdims=True)
    x_std=context[train_idx].std(axis=0,keepdims=True)
    x_std=np.where(x_std<1e-6,1.0,x_std)
    x_scaled=np.nan_to_num((context-x_mean)/x_std,nan=0.0,posinf=0.0,neginf=0.0).astype(np.float32)

    random.seed(SEED+family_index); np.random.seed(SEED+family_index); torch.manual_seed(SEED+family_index)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED+family_index)
    model=ConditionalRealNVP(len(specs),context.shape[1]).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-5)
    dataset=torch.utils.data.TensorDataset(torch.from_numpy(target[train_idx]),torch.from_numpy(x_scaled[train_idx]))
    loader=torch.utils.data.DataLoader(dataset,batch_size=BATCH_SIZE,shuffle=True,drop_last=False)
    valid_t=torch.from_numpy(target[valid_idx]).to(DEVICE)
    valid_x=torch.from_numpy(x_scaled[valid_idx]).to(DEVICE)
    history=[]; best=float('inf'); best_epoch=0; best_state=None; stale=0
    for epoch in range(EPOCHS):
        model.train(); losses=[]
        for yb,xb in loader:
            yb,xb=yb.to(DEVICE),xb.to(DEVICE)
            loss=-model.log_prob(yb,xb).mean()
            optimizer.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); optimizer.step()
            losses.append(float(loss.detach().cpu()))
        model.eval()
        with torch.no_grad(): val_nll=float(-model.log_prob(valid_t,valid_x).mean().cpu())
        train_nll=float(np.mean(losses))
        history.append({'family':family,'epoch':epoch+1,'train_nll':train_nll,'validation_nll':val_nll})
        if val_nll<best-1e-4:
            best=val_nll; best_epoch=epoch+1; stale=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else: stale+=1
        if (epoch+1)%10==0 or epoch==0:
            print(f'Epoch {epoch+1:02d}: train NLL={train_nll:.4f}; validation NLL={val_nll:.4f}; best={best_epoch}')
        if stale>=PATIENCE:
            print(f'[INFO] Early stopping at epoch {epoch+1}; selected epoch {best_epoch}')
            break
    if best_state is None: raise RuntimeError(f'{family} flow training produced no checkpoint')
    model.load_state_dict(best_state); model.eval()
    print(f'[PASS] {family} restored best validation model at epoch {best_epoch}')

    cal_lo,cal_med,cal_hi=posterior_quantiles(model,context,cal_idx,specs,x_scaled,y_mean,y_std,POSTERIOR_DRAWS,SEED+100+family_index)
    test_lo,test_med,test_hi=posterior_quantiles(model,context,test_idx,specs,x_scaled,y_mean,y_std,POSTERIOR_DRAWS,SEED+200+family_index)
    truth_cal=theta[cal_idx]; truth_test=theta[test_idx]
    # Split conformalized quantile regression: calibrate each parameter's lower/upper tail jointly.
    corrections=[]
    calibrated_lo=[]; calibrated_hi=[]
    for j,(name,low,high,scale) in enumerate(specs):
        scores=np.maximum(cal_lo[:,j]-truth_cal[:,j],truth_cal[:,j]-cal_hi[:,j])
        k=min(len(scores),int(np.ceil((len(scores)+1)*(1-ALPHA))))
        correction=float(np.sort(scores)[k-1])
        corrections.append(correction)
        calibrated_lo.append(np.clip(test_lo[:,j]-correction,low,high))
        calibrated_hi.append(np.clip(test_hi[:,j]+correction,low,high))
    calibrated_lo=np.column_stack(calibrated_lo); calibrated_hi=np.column_stack(calibrated_hi)
    for j,(name,low,high,scale) in enumerate(specs):
        raw_cov=float(((truth_test[:,j]>=test_lo[:,j])&(truth_test[:,j]<=test_hi[:,j])).mean())
        cal_cov=float(((truth_test[:,j]>=calibrated_lo[:,j])&(truth_test[:,j]<=calibrated_hi[:,j])).mean())
        cal_raw=float(((truth_cal[:,j]>=cal_lo[:,j])&(truth_cal[:,j]<=cal_hi[:,j])).mean())
        raw_w=float(np.median(test_hi[:,j]-test_lo[:,j]))
        cal_w=float(np.median(calibrated_hi[:,j]-calibrated_lo[:,j]))
        all_rows.append({'family':family,'parameter':name,'nominal_coverage':0.90,
                         'raw_test_coverage':raw_cov,'conformal_test_coverage':cal_cov,
                         'raw_calibration_coverage':cal_raw,'conformal_expansion_parameter_units':corrections[j],
                         'raw_median_interval_width':raw_w,'conformal_median_interval_width':cal_w,
                         'median_width_ratio':cal_w/raw_w if raw_w>0 else None,
                         'test_groups':len(g_test),'calibration_groups':len(g_cal),
                         'prior_lower':low,'prior_upper':high})

    model_path=OUT/f'{family}_conditional_realnvp.pt'
    torch.save({'state_dict':model.cpu().state_dict(),'family':family,'parameter_specs':specs,
                'context_dim':int(context.shape[1]),'context_mean':x_mean.astype(np.float32),
                'context_std':x_std.astype(np.float32),'target_mean':y_mean.astype(np.float32),
                'target_std':y_std.astype(np.float32),'seed':SEED+family_index},model_path)
    summaries[family]={'source_simulation_dir':str(root),'unique_parameter_draws':int(n_draws),
                       'train_groups':int(len(g_train)),'validation_groups':int(len(g_valid)),
                       'calibration_groups':int(len(g_cal)),'test_groups':int(len(g_test)),
                       'best_epoch':int(best_epoch),'best_validation_nll':float(best),
                       'epochs_run':len(history),'posterior_draws_per_calibration_or_test_case':POSTERIOR_DRAWS,
                       'seconds':float(time.time()-family_start),'checkpoint':str(model_path)}
    pd.DataFrame(history).to_csv(OUT/f'{family}_training_history.csv',index=False)
    del model, optimizer, dataset, loader, valid_t, valid_x, context, obs, frame, theta, target, x_scaled
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

results=pd.DataFrame(all_rows)
results.to_csv(OUT/'day16_flow_calibration_by_parameter.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(15,5))
for ax,family in zip(axes,('binary','zeeman')):
    part=results[results.family==family]
    xx=np.arange(len(part)); width=0.36
    ax.bar(xx-width/2,part.raw_test_coverage,width,label='raw flow interval')
    ax.bar(xx+width/2,part.conformal_test_coverage,width,label='conformal interval')
    ax.axhline(0.90,color='black',linestyle='--',linewidth=1,label='nominal 90%')
    ax.set_xticks(xx,part.parameter,rotation=65,ha='right')
    ax.set_ylim(0,1.02); ax.set_ylabel('Held-out empirical coverage')
    ax.set_title(f'{family.title()} — untouched test groups'); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT/'day16_flow_calibration_coverage.png',dpi=160); plt.show()

summary={
    'created_utc':pd.Timestamp.now(tz='UTC').isoformat(),
    'method':'conditional RealNVP plus split conformalized quantile intervals',
    'split_by':'unique parameter_id groups: 70% train, 10% validation, 10% calibration, 10% untouched test',
    'nominal_interval_coverage':0.90,
    'alpha':ALPHA,
    'posterior_draws_per_case_for_quantiles':POSTERIOR_DRAWS,
    'device':str(DEVICE),
    'families':summaries,
    'per_parameter_results':'day16_flow_calibration_by_parameter.csv',
    'interpretation':[
        'Conformal corrections are fit only on parameter groups excluded from model fitting and early stopping; final coverage is measured on a separate untouched test split.',
        'Marginal coverage applies to exchangeable synthetic simulator cases under the tested prior and split; it does not establish conditional coverage at every signal strength or real-star calibration.',
        'Compare conformal interval coverage with its width increase; wider intervals trade precision for coverage.',
        'This notebook uses the saved Day 7 simulators and does not regenerate simulations or run NUTS.'
    ]
}
(OUT/'day16_flow_calibration_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print()
print('Saved Day 16 artifacts under:',OUT)
print(results[['family','parameter','raw_test_coverage','conformal_test_coverage','raw_median_interval_width','conformal_median_interval_width']].to_string(index=False))
print(json.dumps(summary['interpretation'],indent=2))
